# 02 — Preprocessing Pipeline

Full preprocessing pipeline applied to **both** Subject 1 records:
- `AccTempEDA` — ax, ay, az, temp, EDA @ 8 Hz
- `SpO2HR` — SpO2, hr @ 1 Hz

Steps per record:
1. Low-pass Butterworth filter (zero-phase)
2. Per-subject Z-score normalization (stats over full recording, before segmentation)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from src.preprocessing import load_record, filter_signal, normalize_record, subject_paths

## Load Subject 1 — both records

In [ ]:
eda_path, hr_path = subject_paths(1)

rec_eda = load_record(eda_path, ann_ext='atr')  # has annotations
rec_hr  = load_record(hr_path,  ann_ext=None)   # no annotation file

# SpO2HR has no annotation file — copy phase boundaries from AccTempEDA, rescaled to 1 Hz
boundary_s = [s / rec_eda['fs'] for s in rec_eda['ann_samples']]
rec_hr['ann_samples'] = [round(t * rec_hr['fs']) for t in boundary_s]
rec_hr['ann_labels']  = rec_eda['ann_labels']

for name, rec in [('AccTempEDA', rec_eda), ('SpO2HR', rec_hr)]:
    dur = rec['signal'].shape[0] / rec['fs'] / 60
    print(f"{name}")
    print(f"  channels : {rec['sig_name']}")
    print(f"  fs       : {rec['fs']} Hz")
    print(f"  shape    : {rec['signal'].shape}  (samples x channels)")
    print(f"  duration : {dur:.1f} min")
    print()

## Apply Low-Pass Butterworth Filter

| Record | fs | Nyquist | Cutoff |
|---|---|---|---|
| AccTempEDA | 8 Hz | 4 Hz | 1.0 Hz |
| SpO2HR | 1 Hz | 0.5 Hz | 0.1 Hz |

Both use order-4, zero-phase (`filtfilt`).

In [ ]:
sig_eda_filt = filter_signal(rec_eda['signal'], fs=rec_eda['fs'], cutoff=1.0)
sig_hr_filt  = filter_signal(rec_hr['signal'],  fs=rec_hr['fs'],  cutoff=0.1)

rec_eda_filt = {**rec_eda, 'signal': sig_eda_filt}
rec_hr_filt  = {**rec_hr,  'signal': sig_hr_filt}

print("AccTempEDA: low-pass at 1.0 Hz (order 4, zero-phase)")
print("SpO2HR    : low-pass at 0.1 Hz (order 4, zero-phase)")

## Before vs After — AccTempEDA (8 Hz, cutoff 1.0 Hz)

Raw signal in blue, filtered in orange. Phase labels on top subplot.

In [ ]:
def plot_before_after(rec_raw, sig_filt, title):
    fs = rec_raw['fs']
    channel_names = rec_raw['sig_name']
    num_channels = len(channel_names)
    time_axis = np.arange(rec_raw['signal'].shape[0]) / fs

    # Convert phase boundary sample indices to seconds
    phase_times  = [s / fs for s in rec_raw['ann_samples']]
    phase_labels = rec_raw['ann_labels']

    fig, axes = plt.subplots(num_channels, 1, figsize=(14, 2.5 * num_channels), sharex=True)
    if num_channels == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=13)

    for channel_idx, ax in enumerate(axes):
        # Overlay raw and filtered signals
        ax.plot(time_axis, rec_raw['signal'][:, channel_idx], lw=0.8, alpha=0.6, label='raw')
        ax.plot(time_axis, sig_filt[:, channel_idx], lw=0.8, label='filtered')

        # Draw a vertical red line at each phase boundary
        for phase_time, phase_label in zip(phase_times, phase_labels):
            ax.axvline(phase_time, color='red', lw=0.8, alpha=0.6)

            # Only label phases on the top subplot to avoid clutter
            if channel_idx == 0:
                ax.text(phase_time + 1, ax.get_ylim()[1], phase_label,
                        fontsize=7, color='red', rotation=45)

        ax.set_ylabel(channel_names[channel_idx])
        ax.legend(loc='upper right', fontsize=7)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('Time (s)')
    plt.tight_layout()
    plt.show()

plot_before_after(rec_eda, sig_eda_filt, 'Subject 1 — AccTempEDA: Raw vs Filtered (cutoff=1.0 Hz)')

## Before vs After — SpO2HR (1 Hz, cutoff 0.1 Hz)

Raw signal in blue, filtered in orange. Phase labels on top subplot.

In [ ]:
plot_before_after(rec_hr, sig_hr_filt, 'Subject 1 — SpO2HR: Raw vs Filtered (cutoff=0.1 Hz)')

## Apply Per-Subject Z-Score Normalization

Stats (mean, std) computed over the entire recording before segmentation.

In [ ]:
rec_eda_norm = normalize_record(rec_eda_filt)
rec_hr_norm  = normalize_record(rec_hr_filt)

for rec_label, rec_before, rec_after in [
    ('AccTempEDA', rec_eda_filt, rec_eda_norm),
    ('SpO2HR',     rec_hr_filt,  rec_hr_norm),
]:
    print(f"\n{rec_label}")
    print(f"  {'Channel':<8} {'mean before':>12} {'std before':>11} {'mean after':>11} {'std after':>10}")
    print('  ' + '-' * 55)
    for i, ch in enumerate(rec_before['sig_name']):
        mb = rec_before['signal'][:, i].mean()
        sb = rec_before['signal'][:, i].std()
        ma = rec_after['signal'][:, i].mean()
        sa = rec_after['signal'][:, i].std()
        print(f"  {ch:<8} {mb:>12.4f} {sb:>11.4f} {ma:>11.6f} {sa:>10.6f}")

## Normalized Signals — All Phases

EDA, accelerometer magnitude, skin temperature, HR, and SpO2 after full preprocessing (filter → z-score).

In [ ]:
sig_e = rec_eda_norm['signal']
sig_h = rec_hr_norm['signal']
chs_e = rec_eda_norm['sig_name']
chs_h = rec_hr_norm['sig_name']

# Build time axes and derived signals
t_eda   = np.arange(len(sig_e)) / rec_eda_norm['fs']
t_hr    = np.arange(len(sig_h)) / rec_hr_norm['fs']
acc_mag = np.sqrt(sig_e[:, chs_e.index('ax')]**2 +
                  sig_e[:, chs_e.index('ay')]**2 +
                  sig_e[:, chs_e.index('az')]**2)

plot_specs = [
    (t_eda, sig_e[:, chs_e.index('EDA')],  'EDA (z-score)',           'steelblue',    rec_eda_norm),
    (t_eda, acc_mag,                         'Acc magnitude (z-score)', 'darkorange',   rec_eda_norm),
    (t_eda, sig_e[:, chs_e.index('temp')],  'Temp (z-score)',          'firebrick',    rec_eda_norm),
    (t_hr,  sig_h[:, chs_h.index('hr')],   'HR (z-score)',            'seagreen',     rec_hr_norm),
    (t_hr,  sig_h[:, chs_h.index('SpO2')], 'SpO2 (z-score)',          'mediumpurple', rec_hr_norm),
]

fig, axes = plt.subplots(len(plot_specs), 1, figsize=(14, 2.5 * len(plot_specs)), sharex=False)
fig.suptitle('Subject 1 — Normalized signals: filter + z-score (all phases)', fontsize=13)

for subplot_idx, (ax, (t, sig, ylabel, color, rec)) in enumerate(zip(axes, plot_specs)):
    ax.plot(t, sig, color=color, lw=0.8)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Time (s)')
    ax.grid(True, alpha=0.3)

    # Draw a vertical red line at each phase boundary
    phase_times  = [s / rec['fs'] for s in rec['ann_samples']]
    phase_labels = rec['ann_labels']
    for phase_time, phase_label in zip(phase_times, phase_labels):
        ax.axvline(phase_time, color='red', lw=0.8, alpha=0.6)

        # Only label phases on the top subplot of each record group
        if subplot_idx in (0, 3):
            ax.text(phase_time + 1, ax.get_ylim()[1], phase_label,
                    fontsize=7, color='red', rotation=45)

plt.tight_layout()
plt.show()